# 04 — Few-Shot Training (Phase B)

Prototypical Network training on FineBadminton labeled data
with SSL pre-trained encoder from Phase A.

**Player ordering:** `FineBadmintonDataset` automatically reorders each skeleton so
the **hitting player is at nodes 0–16** and the opponent at nodes 17–33,
using the `"hitter"` field ("top"/"bottom") from the annotations.
This ensures the model always sees the strategically relevant player first.

**Pipeline:**
1. Load SSL pre-trained encoder (feature-layer aware)
2. Load FineBadminton labeled data with enriched features (L0–L3), hitter-first
3. 5-fold stratified CV with episodic sampling
4. Train ProtoNet (+ optional fine-tuning of encoder)
5. Evaluate: macro-F1, per-class F1, confusion matrix
6. Alternative classifiers: k-NN, linear probe
7. Save best prototypes and results

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from pathlib import Path
from functools import partial

from src.config import *
from src.data.graph_builder import GraphBuilder
from src.data.dataset import FineBadmintonDataset, EpisodicSampler, collate_episode
from src.models.stgcn_model import STGCN
from src.models.transformer_encoder import SkeletonTransformer
from src.models.proto_net import PrototypicalNetwork

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {device}")

## 1. Configuration

In [ ]:
cfg = get_config('fewshot')

# Feature layer — must match what was used for SSL pre-training
FEATURE_LAYER = 'L2'
feature_dim = FEATURE_DIMS[FEATURE_LAYER]
cfg.stgcn.in_channels = feature_dim

print(f"Feature layer: {FEATURE_LAYER} ({feature_dim} features/node)")
print(f"\nEncoder: ST-GCN")
print(f"  in_channels: {cfg.stgcn.in_channels}")
print(f"  num_nodes: {cfg.stgcn.num_nodes}")
print(f"  embedding_dim: {cfg.stgcn.embedding_dim}")
print(f"\nProtoNet:")
print(f"  n_way: {cfg.proto.n_way}")
print(f"  k_shot: {cfg.proto.k_shot}")
print(f"  n_query: {cfg.proto.n_query}")
print(f"  distance: {cfg.proto.distance}")
print(f"  episodes_per_epoch: {cfg.proto.episodes_per_epoch}")
print(f"  epochs: {cfg.proto.epochs}")
print(f"  fine_tune_encoder: {cfg.proto.fine_tune_encoder}")

## 2. Build Encoder and Load Pre-trained Weights

In [ ]:
# Build graph adjacency
graph_builder = GraphBuilder(
    use_inter_player=cfg.ablation.use_inter_player,
    single_player=cfg.ablation.single_player,
)
adjacency = graph_builder.build_adjacency().to(device)
print(f"Adjacency: {adjacency.shape}")

# Build encoder
encoder = STGCN(
    in_channels=cfg.stgcn.in_channels,
    num_nodes=cfg.stgcn.num_nodes,
    adjacency=adjacency,
    num_layers=cfg.stgcn.num_layers,
    base_channels=cfg.stgcn.base_channels,
    embedding_dim=cfg.stgcn.embedding_dim,
    temporal_kernel=cfg.stgcn.temporal_kernel,
    dropout=cfg.stgcn.dropout,
).to(device)

total_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {total_params:,}")

# Load SSL pre-trained weights
ssl_path = MODELS_DIR / f'ssl_pretrained_{FEATURE_LAYER}.pt'
ssl_checkpoint = None

if ssl_path.exists() and cfg.ablation.use_pretrained:
    ssl_checkpoint = torch.load(ssl_path, map_location=device, weights_only=False)
    encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])
    print(f"Loaded SSL weights from {ssl_path}")
    print(f"  SSL feature layer: {ssl_checkpoint.get('feature_layer', 'unknown')}")
    if 'history' in ssl_checkpoint:
        final_loss = ssl_checkpoint['history']['loss'][-1]
        print(f"  Final SSL loss: {final_loss:.4f}")
else:
    print(f"No SSL weights found at {ssl_path}")
    print("Using randomly initialized encoder")

## 3. Load FineBadminton Data

In [ ]:
dataset = FineBadmintonDataset(
    shot_window=cfg.data.shot_window,
    feature_layer=FEATURE_LAYER,
)
print(f"Dataset size: {len(dataset)}")

# Show class distribution
from collections import Counter
label_dist = Counter(dataset.get_labels())
print(f"\nClass distribution:")
for idx in sorted(label_dist.keys()):
    print(f"  {IDX_TO_STRATEGY[idx]}: {label_dist[idx]}")

# Get cross-validation splits
splits = dataset.get_fold_splits(n_folds=cfg.data.num_folds, seed=cfg.data.random_seed)
print(f"\nFolds: {len(splits)}")
for i, (train, val, test) in enumerate(splits):
    print(f"  Fold {i+1}: train={len(train)}, val={len(val)}, test={len(test)}")

## 4. Helper Functions

In [ ]:
def extract_embeddings(encoder, dataset, indices, device):
    """Extract embeddings for a subset of the dataset."""
    encoder.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for i in indices:
            x, y = dataset[i]
            x = x.unsqueeze(0).to(device)
            emb = encoder(x).cpu().numpy()[0]
            embeddings.append(emb)
            labels.append(y)
    return np.array(embeddings), np.array(labels)


def train_one_epoch(encoder, proto_net, dataset, train_idx, optimizer, cfg, device):
    """Train one epoch of episodic training."""
    encoder.train()
    train_labels = [dataset.get_labels()[i] for i in train_idx]
    
    sampler = EpisodicSampler(
        labels=train_labels,
        n_way=cfg.proto.n_way,
        k_shot=cfg.proto.k_shot,
        n_query=cfg.proto.n_query,
        episodes_per_epoch=cfg.proto.episodes_per_epoch,
    )
    
    epoch_loss = 0.0
    epoch_acc = 0.0
    n_episodes = 0
    
    for episode_indices in sampler:
        # Map local indices back to dataset indices
        actual_indices = [train_idx[i] for i in episode_indices]
        
        # Load episode data
        batch = [dataset[i] for i in actual_indices]
        support_x, support_y, query_x, query_y = collate_episode(
            batch, cfg.proto.n_way, cfg.proto.k_shot, cfg.proto.n_query
        )
        
        support_x = support_x.to(device)
        support_y = support_y.to(device)
        query_x = query_x.to(device)
        query_y = query_y.to(device)
        
        # Forward pass
        support_emb = encoder(support_x)
        query_emb = encoder(query_x)
        
        # Compute prototypes
        prototypes = proto_net.compute_prototypes(support_emb, support_y)
        
        # Compute distances and loss
        distances = proto_net.compute_distances(query_emb, prototypes)
        log_probs = torch.nn.functional.log_softmax(-distances, dim=1)
        loss = torch.nn.functional.nll_loss(log_probs, query_y)
        
        # Accuracy
        preds = (-distances).argmax(dim=1)
        acc = (preds == query_y).float().mean().item()
        
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        epoch_loss += loss.item()
        epoch_acc += acc
        n_episodes += 1
    
    return epoch_loss / n_episodes, epoch_acc / n_episodes


def evaluate_protonet(encoder, dataset, support_idx, test_idx, device):
    """Evaluate ProtoNet: compute prototypes from support, classify test."""
    support_emb, support_labels = extract_embeddings(encoder, dataset, support_idx, device)
    test_emb, test_labels = extract_embeddings(encoder, dataset, test_idx, device)
    
    # Compute prototypes (class centroids)
    prototypes = {}
    for c in np.unique(support_labels):
        prototypes[c] = support_emb[support_labels == c].mean(axis=0)
    
    # Classify by nearest prototype
    preds = []
    for emb in test_emb:
        dists = {c: np.linalg.norm(emb - p) for c, p in prototypes.items()}
        preds.append(min(dists, key=dists.get))
    preds = np.array(preds)
    
    macro_f1 = f1_score(test_labels, preds, average='macro')
    per_class = f1_score(test_labels, preds, average=None)
    cm = confusion_matrix(test_labels, preds)
    report = classification_report(test_labels, preds, 
                                    target_names=[IDX_TO_STRATEGY[i] for i in sorted(prototypes.keys())],
                                    output_dict=True)
    
    return {
        'macro_f1': macro_f1,
        'per_class_f1': per_class.tolist(),
        'confusion_matrix': cm,
        'report': report,
        'prototypes': prototypes,
        'predictions': preds,
        'true_labels': test_labels,
    }

## 5. Episodic Training Loop (5-Fold CV)

In [ ]:
all_fold_results = []
training_history = []

for fold_idx, (train_idx, val_idx, test_idx) in enumerate(splits):
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx + 1}/{len(splits)}")
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    
    # Reset encoder to SSL pre-trained weights for each fold
    if ssl_checkpoint is not None:
        encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])
    else:
        # Re-init for fair comparison across folds
        encoder = STGCN(
            in_channels=cfg.stgcn.in_channels,
            num_nodes=cfg.stgcn.num_nodes,
            adjacency=adjacency,
            num_layers=cfg.stgcn.num_layers,
            base_channels=cfg.stgcn.base_channels,
            embedding_dim=cfg.stgcn.embedding_dim,
            temporal_kernel=cfg.stgcn.temporal_kernel,
            dropout=cfg.stgcn.dropout,
        ).to(device)
    
    # Build prototypical network
    proto_net = PrototypicalNetwork(encoder, distance=cfg.proto.distance)
    
    # Setup optimizer
    if cfg.proto.fine_tune_encoder:
        optimizer = optim.Adam(
            encoder.parameters(),
            lr=cfg.proto.lr * cfg.proto.encoder_lr_scale,
            weight_decay=1e-5,
        )
    else:
        # Freeze encoder
        encoder.eval()
        for p in encoder.parameters():
            p.requires_grad = False
        optimizer = None
    
    # Training loop
    fold_history = {'train_loss': [], 'train_acc': [], 'val_f1': []}
    best_val_f1 = 0.0
    best_encoder_state = None
    
    for epoch in range(cfg.proto.epochs):
        train_loss, train_acc = train_one_epoch(
            encoder, proto_net, dataset, train_idx, optimizer, cfg, device
        )
        fold_history['train_loss'].append(train_loss)
        fold_history['train_acc'].append(train_acc)
        
        # Validate every 5 epochs
        if (epoch + 1) % 5 == 0:
            val_result = evaluate_protonet(encoder, dataset, train_idx, val_idx, device)
            fold_history['val_f1'].append(val_result['macro_f1'])
            
            if val_result['macro_f1'] > best_val_f1:
                best_val_f1 = val_result['macro_f1']
                best_encoder_state = {k: v.clone() for k, v in encoder.state_dict().items()}
            
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}/{cfg.proto.epochs} — "
                      f"Loss: {train_loss:.4f}, Acc: {train_acc:.3f}, "
                      f"Val-F1: {val_result['macro_f1']:.3f}")
    
    # Restore best model and evaluate on test set
    if best_encoder_state is not None:
        encoder.load_state_dict(best_encoder_state)
    
    test_result = evaluate_protonet(encoder, dataset, train_idx, test_idx, device)
    
    print(f"\n  Test Macro-F1: {test_result['macro_f1']:.3f}")
    print(f"  Per-class F1: {[f'{f:.3f}' for f in test_result['per_class_f1']]}")
    print(f"  Best Val-F1: {best_val_f1:.3f}")
    
    fold_result = {
        'fold': fold_idx,
        'macro_f1': test_result['macro_f1'],
        'per_class_f1': test_result['per_class_f1'],
        'confusion_matrix': test_result['confusion_matrix'],
        'report': test_result['report'],
        'best_val_f1': best_val_f1,
        'prototypes': test_result['prototypes'],
    }
    all_fold_results.append(fold_result)
    training_history.append(fold_history)

# Summary
f1_scores = [r['macro_f1'] for r in all_fold_results]
print(f"\n{'='*60}")
print(f"Overall Macro-F1: {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}")
print(f"Per-fold: {[f'{f:.3f}' for f in f1_scores]}")

---
## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, hist in enumerate(training_history):
    axes[0].plot(hist['train_loss'], alpha=0.7, label=f'Fold {i+1}')
    axes[1].plot(hist['train_acc'], alpha=0.7, label=f'Fold {i+1}')
    val_epochs = list(range(4, len(hist['train_loss']), 5))
    axes[2].plot(val_epochs, hist['val_f1'], 'o-', alpha=0.7, label=f'Fold {i+1}')

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Episodic Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Episode Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro-F1')
axes[2].set_title('Validation Macro-F1')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Few-Shot Training ({FEATURE_LAYER} features)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fewshot_training_curves.png', dpi=150)
plt.show()

---
## 7. Confusion Matrix

In [ ]:
# Averaged confusion matrix across folds
avg_cm = np.mean([r['confusion_matrix'] for r in all_fold_results], axis=0)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    avg_cm, annot=True, fmt='.1f', cmap='Blues',
    xticklabels=STRATEGY_CLASSES,
    yticklabels=STRATEGY_CLASSES,
    ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Averaged Confusion Matrix (5-Fold)\nMacro-F1: {np.mean(f1_scores):.3f} ± {np.std(f1_scores):.3f}')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fewshot_confusion_matrix.png', dpi=150)
plt.show()

# Per-class F1 summary
per_class_f1s = np.array([r['per_class_f1'] for r in all_fold_results])
print("Per-class F1 (mean ± std):")
for i, name in enumerate(STRATEGY_CLASSES):
    if i < per_class_f1s.shape[1]:
        print(f"  {name:15s}: {per_class_f1s[:, i].mean():.3f} ± {per_class_f1s[:, i].std():.3f}")

---
## 8. Alternative Classifiers (k-NN, Linear Probe)

Compare ProtoNet against k-NN and logistic regression
using the same frozen encoder embeddings.

In [ ]:
# Use the best encoder from the last fold (or reload SSL weights)
if ssl_checkpoint is not None:
    encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])

# Extract all embeddings once
all_emb, all_labels = extract_embeddings(
    encoder, dataset, list(range(len(dataset))), device
)
print(f"Embeddings: {all_emb.shape}")

# Compare classifiers across folds
classifier_results = {
    'protonet': [],
    'knn_3': [],
    'knn_5': [],
    'linear_probe': [],
}

for fold_idx, (train_idx, val_idx, test_idx) in enumerate(splits):
    X_train = all_emb[train_idx]
    y_train = all_labels[train_idx]
    X_test = all_emb[test_idx]
    y_test = all_labels[test_idx]
    
    # ProtoNet (nearest centroid)
    prototypes = {}
    for c in np.unique(y_train):
        prototypes[c] = X_train[y_train == c].mean(axis=0)
    proto_preds = np.array([
        min(prototypes, key=lambda c: np.linalg.norm(x - prototypes[c]))
        for x in X_test
    ])
    classifier_results['protonet'].append(
        f1_score(y_test, proto_preds, average='macro')
    )
    
    # k-NN (k=3)
    knn3 = KNeighborsClassifier(n_neighbors=3)
    knn3.fit(X_train, y_train)
    classifier_results['knn_3'].append(
        f1_score(y_test, knn3.predict(X_test), average='macro')
    )
    
    # k-NN (k=5)
    knn5 = KNeighborsClassifier(n_neighbors=5)
    knn5.fit(X_train, y_train)
    classifier_results['knn_5'].append(
        f1_score(y_test, knn5.predict(X_test), average='macro')
    )
    
    # Linear probe (logistic regression)
    lr = LogisticRegression(max_iter=1000, C=1.0)
    lr.fit(X_train, y_train)
    classifier_results['linear_probe'].append(
        f1_score(y_test, lr.predict(X_test), average='macro')
    )

# Summary table
print(f"\n{'Method':<15} {'Macro-F1':>12} {'Std':>8}")
print("-" * 38)
for method, f1s in classifier_results.items():
    print(f"{method:<15} {np.mean(f1s):>12.3f} {np.std(f1s):>8.3f}")

---
## 9. Save Results and Best Model

In [ ]:
# Save best fold prototypes and encoder for inference
best_fold = max(range(len(all_fold_results)), key=lambda i: all_fold_results[i]['macro_f1'])
best_result = all_fold_results[best_fold]
print(f"Best fold: {best_fold + 1} (Macro-F1: {best_result['macro_f1']:.3f})")

# Convert prototypes to tensors for saving
proto_tensors = {c: torch.tensor(p) for c, p in best_result['prototypes'].items()}

checkpoint = {
    'encoder_state_dict': encoder.state_dict(),
    'prototypes': proto_tensors,
    'feature_layer': FEATURE_LAYER,
    'config': cfg,
    'fold_results': [
        {k: v.tolist() if isinstance(v, np.ndarray) else v
         for k, v in r.items() if k != 'prototypes'}
        for r in all_fold_results
    ],
    'classifier_comparison': {
        method: {'mean': float(np.mean(f1s)), 'std': float(np.std(f1s)), 'folds': f1s}
        for method, f1s in classifier_results.items()
    },
}

save_path = MODELS_DIR / f'fewshot_{FEATURE_LAYER}.pt'
torch.save(checkpoint, save_path)
print(f"Saved to {save_path}")

# Also save results as JSON for analysis notebook
json_results = {
    'feature_layer': FEATURE_LAYER,
    'overall_macro_f1': float(np.mean(f1_scores)),
    'overall_std': float(np.std(f1_scores)),
    'fold_f1s': [float(f) for f in f1_scores],
    'classifier_comparison': checkpoint['classifier_comparison'],
}
with open(RESULTS_DIR / 'fewshot_results.json', 'w') as f:
    json.dump(json_results, f, indent=2)
print(f"Results saved to {RESULTS_DIR / 'fewshot_results.json'}")